In [3]:
#
import pandas as pd
from pathlib import Path

ruta = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed')
df = pd.read_parquet(ruta / 'df_alumno.parquet')
print(df.columns.tolist())
print(df.shape)
print(df.head(3))

['per_id_ficticio', 'exp_tit_id', 'curso_aca_ini', 'curso_aca', 'curso_aca_fin', 'nota', 'via_acceso_exp', 'seguro', 'nota_selectividad', 'nota_acceso', 'cred_matriculados', 'cred_superados', 'egresado', 'nuevo', 'media_curso', 'titulacion', 'rama', 'cred_titulacion', 'sexo', 'fecha_nacimiento', 'id_pais', 'pais_nombre', 'poblacion', 'provincia', 'pais_domicilio', 'vive_fuera', 'nombre_beca', 'tiene_beca', 'nombre_trabajo', 'media_titulacion_curso', 'media_titulacion_alumno', 'forma_de_pago', 'numero_pagos', 'via_acceso', 'orden_preferencia', 'cupo', 'universidad_origen']
(109568, 37)
   per_id_ficticio  exp_tit_id  curso_aca_ini  curso_aca  curso_aca_fin  nota  \
0              106         223           2012       2012            NaN   NaN   
1              134         209           2009       2012            NaN   NaN   
2              134         209           2009       2010            NaN   NaN   

  via_acceso_exp seguro  nota_selectividad  nota_acceso  ...  tiene_beca  \
0      

Perfecto. Tenemos el output real. Ahora analizo qué hay útil y qué no.

LO QUE TENEMOS EN df_alumno — análisis columna por columna
Identificadores — imprescindibles para el cruce:

per_id_ficticio ✅ — el identificador del alumno, es la clave del cruce
exp_tit_id ✅ — identificador del expediente por titulación

Variables de titulación y académicas:

titulacion ✅ — lo que queremos recuperar
rama ✅ — ya la tienes en X_test, pero la recuperamos igualmente para verificar coherencia
cred_titulacion — créditos totales de la titulación, útil para contexto
curso_aca_ini ✅ — año de ingreso, lo necesitamos para la regla de 2+ titulaciones y para análisis temporal
curso_aca_fin — año de fin, puede ser NaN si no terminó
egresado — si terminó la carrera, interesante para FP/FN
media_titulacion_curso — media del curso en esa titulación, contexto
media_titulacion_alumno — media del alumno en esa titulación

Variables personales — útiles para fairness:

sexo ✅ — ya la tienes en X_test
fecha_nacimiento — de aquí se puede calcular edad real, pero ya tienes edad_entrada
pais_nombre ✅ — ya la tienes en X_test
id_pais, poblacion, provincia, pais_domicilio — contexto geográfico
vive_fuera ✅ — interesante: ¿vivir fuera de la ciudad aumenta el abandono? No está en X_test

Variables de acceso:

via_acceso ✅ — ya la tienes en X_test
via_acceso_exp — versión alternativa de vía de acceso, puede tener más categorías
orden_preferencia ✅ — ya la tienes en X_test
cupo — tipo de cupo de acceso (General, Discapacidad, Deportistas...) — no está en X_test, interesante para fairness
universidad_origen ✅ — ya la tienes en X_test

Variables económicas:

nombre_beca, tiene_beca — versión alternativa de beca, ya tienes tuvo_beca y n_anios_beca en X_test
forma_de_pago, numero_pagos — patrón de pago de matrícula, no está en X_test
seguro — seguro escolar, poco relevante

Variables de trabajo:

nombre_trabajo — tipo de trabajo si trabaja, ya tienes situacion_laboral en X_test


LO QUE RECUPERAMOS PARA meta_test.parquet
Variables nuevas que NO están en X_test y son útiles:
VariablePara quétitulacionSHAP cohortes, fairness, FP/FN — la más importantecurso_aca_iniAnálisis temporal — cohorte de ingresovive_fueraFairness — ¿penaliza el modelo vivir fuera?cupoFairness — acceso por discapacidad, deportistas, etc.egresadoContexto FP/FN — ¿los falsos negativos terminaron la carrera?via_acceso_expVersión más granular de vía de acceso
Las que ya tienes en X_test (sexo, pais_nombre, rama, via_acceso, etc.) no hace falta recuperarlas — ya están.

LO QUE HEMOS HECHO hasta ahora en esta sesión
Solo exploración — no hemos tocado ningún fichero ni notebook. Hemos confirmado:

df_alumno.parquet existe, tiene 37 columnas, 109.568 filas
Tiene per_id_ficticio como identificador del alumno
Tiene titulacion, curso_aca_ini, vive_fuera, cupo, egresado — recuperables
El índice de X_test son posiciones en dataset_final_tfm (0–33.620)
dataset_final_tfm NO tiene per_id_ficticio — se perdió en Fase 3

No hemos modificado nada todavía.

In [5]:
#El siguiente paso es encontrar el notebook de Fase 3 que genera dataset_final_tfm.parquet
from pathlib import Path

ruta = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks')
for f in sorted(ruta.rglob('*.ipynb')):
    if 'f3' in f.name.lower() or 'fase3' in f.name.lower() or 'feature' in f.name.lower():
        print(f.name)

f3_m00_indice.ipynb
f3_m01_validacion.ipynb
f3_m02_agregacion.ipynb
f3_m03_features.ipynb
f3_m04_index.ipynb
f3_m04a_automl_target.ipynb
f3_m04b_eda_target.ipynb
f3_m05_target_export.ipynb
f3_m06_alerta_temprana.ipynb
f3_m07_validacion.ipynb
f3_m08_auditoria.ipynb
f4_m07_seleccion_features.ipynb


In [4]:
#alumnos en eldatsest
import pandas as pd
from pathlib import Path

ruta = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed')
df = pd.read_parquet(ruta / 'df_alumno.parquet')

# Alumnos únicos
n_alumnos = df['per_id_ficticio'].nunique()
print(f"Alumnos únicos: {n_alumnos}")

# Filas totales
print(f"Filas totales (alumno x año x titulación): {len(df)}")

# Titulaciones por alumno
n_tit = df.groupby('per_id_ficticio')['titulacion'].nunique()
print(f"\nDistribución titulaciones por alumno:")
print(n_tit.value_counts().sort_index())
print(f"\nAlumnos con 2+ titulaciones: {(n_tit > 1).sum()} ({(n_tit > 1).mean()*100:.1f}%)")

# Años disponibles
print(f"\nCursos académicos: {sorted(df['curso_aca_ini'].dropna().unique().tolist())}")

Alumnos únicos: 30872
Filas totales (alumno x año x titulación): 109568

Distribución titulaciones por alumno:
titulacion
1    28279
2     2472
3      116
4        4
5        1
Name: count, dtype: int64

Alumnos con 2+ titulaciones: 2593 (8.4%)

Cursos académicos: [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]


In [6]:
#El notebook que genera dataset_final_tfm.parquet es casi seguro f3_m05_target_export.ipynb — el nombre lo dice: target + export.
from pathlib import Path

ruta = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks')

# Buscar en todos los notebooks de Fase 3 cuál menciona dataset_final_tfm
for f in sorted(ruta.rglob('f3*.ipynb')):
    contenido = f.read_text(encoding='utf-8', errors='ignore')
    if 'dataset_final_tfm' in contenido:
        print(f"✅ {f.name}")
    else:
        print(f"   {f.name}")

   f3_m00_indice.ipynb
   f3_m01_validacion.ipynb
   f3_m02_agregacion.ipynb
   f3_m03_features.ipynb
   f3_m04_index.ipynb
   f3_m04a_automl_target.ipynb
   f3_m04b_eda_target.ipynb
   f3_m05_target_export.ipynb
   f3_m06_alerta_temprana.ipynb
   f3_m07_validacion.ipynb
✅ f3_m08_auditoria.ipynb


In [ ]:
#
from pathlib import Path
import json

ruta = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks')
nb = json.loads((ruta / 'f3_m08_auditoria.ipynb').read_text(encoding='utf-8'))

for i, cell in enumerate(nb['cells']):
    src = ''.join(cell['source'])
    if 'dataset_final_tfm' in src:
        print(f"\n--- Celda {i} ({cell['cell_type']}) ---")
        print(src[:500])

In [7]:
import pandas as pd
from pathlib import Path

ruta = Path(r'C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\04_eda')
df_eda = pd.read_parquet(ruta / 'df_eda_final.parquet')
print(df_eda.shape)
print(df_eda.columns.tolist())
print(df_eda.head(3))

(33621, 21)
['anios_gap', 'anios_sin_beca', 'cred_superados_anio_1er', 'edad_entrada', 'indicador_interrupcion', 'max_pagos', 'n_anios_beca', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'orden_preferencia', 'pais_nombre', 'provincia', 'rama', 'sexo', 'situacion_laboral', 'tuvo_beca', 'universidad_origen', 'via_acceso', 'titulacion', 'abandono']
   anios_gap  anios_sin_beca  cred_superados_anio_1er  edad_entrada  \
0          0               1                     12.0            38   
1          0               2                    120.0            36   
2          0               2                     60.0            41   

   indicador_interrupcion  max_pagos  n_anios_beca  nota_1er_anio  \
0                       0        4.0             0           5.95   
1                       0        4.0             1           6.56   
2                       0        2.0             2           7.93   

   nota_acceso  nota_selectividad  ...  pais_nombre provincia rama    sexo  \
0   